# Sparsity Preserving QAT

## Install TensorFlow Model Optimization Toolkit
* 설치 완료 후 반드시 Runtime 재시작!
    * '런타임' > '세션 다시 시작' 메뉴 선택

In [ ]:
!pip install tensorflow-model-optimization

## Mount Google driver

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('g-drive mounted.')
    colab=True
except:
    print('local drive.')
    colab =False

Mounted at /content/drive
g-drive mounted.


In [ ]:
if colab :
  save_dir = '/content/drive/MyDrive/files/save/'
else :
  save_dir = '../files/save/'

## Import Module

In [ ]:
import tensorflow as tf
import numpy as np

import tensorflow_model_optimization as tfmot
from tensorflow_model_optimization.python.core.keras.compat import keras

import tempfile

print(tf.__version__)
print(np.__version__)

2.19.0
1.26.4


## Load Dataset

In [ ]:
(train_images, train_labels), (test_images, test_labels) = keras.datasets.mnist.load_data()

train_images = (train_images / 255.0).astype(np.float32)
test_images = (test_images / 255.0).astype(np.float32)

11490434/11490434 [==============================] - 0s 0us/step


In [ ]:
train_images = np.expand_dims(train_images, axis=-1)
test_images = np.expand_dims(test_images, axis=-1)

## Load Baseline Model for MNIST

In [ ]:
model = keras.models.load_model(save_dir + 'baseline_model.h5')
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 26, 26, 32)        320       
                                                                 
 max_pooling2d (MaxPooling2  (None, 13, 13, 32)        0         
 D)                                                              
                                                                 
 conv2d_1 (Conv2D)           (None, 11, 11, 16)        4624      
                                                                 
 max_pooling2d_1 (MaxPoolin  (None, 5, 5, 16)          0         
 g2D)                                                            
                                                                 
 flatten (Flatten)           (None, 400)               0         
                                                                 
 dense (Dense)               (None, 128)               5

In [ ]:
_, baseline_model_accuracy = model.evaluate(
    test_images, test_labels, verbose=0)

print('Baseline test accuracy:', baseline_model_accuracy)

Baseline test accuracy: 0.9904000163078308


## Prune and fine-tune
* **purning schedule** : Constant Sparsity
    - sparsity : 0.5
    - begin_step : 0
    - frequency : every 100 step


In [ ]:
pruning_params = {
      'pruning_schedule': tfmot.sparsity.keras.ConstantSparsity(0.5, begin_step=0, frequency=100)
  }

callbacks = [
  tfmot.sparsity.keras.UpdatePruningStep()
]

pruned_model = tfmot.sparsity.keras.prune_low_magnitude(model, **pruning_params)

pruned_model.compile(
  loss=keras.losses.SparseCategoricalCrossentropy(),
  optimizer=keras.optimizers.Adam(learning_rate=1e-5), # 작은 learning rate 적용
  metrics=['accuracy'])

pruned_model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 prune_low_magnitude_conv2d  (None, 26, 26, 32)        610       
  (PruneLowMagnitude)                                            
                                                                 
 prune_low_magnitude_max_po  (None, 13, 13, 32)        1         
 oling2d (PruneLowMagnitude                                      
 )                                                               
                                                                 
 prune_low_magnitude_conv2d  (None, 11, 11, 16)        9234      
 _1 (PruneLowMagnitude)                                          
                                                                 
 prune_low_magnitude_max_po  (None, 5, 5, 16)          1         
 oling2d_1 (PruneLowMagnitu                                      
 de)                                                    

## Fine tune the model for pruning

In [ ]:
pruned_model.fit(
  train_images,
  train_labels,
  epochs=3,
  validation_split=0.1,
  callbacks=callbacks)

Epoch 1/3
1688/1688 [==============================] - 17s 7ms/step - loss: 0.0195 - accuracy: 0.9946 - val_loss: 0.0392 - val_accuracy: 0.9897
Epoch 2/3
1688/1688 [==============================] - 11s 6ms/step - loss: 0.0118 - accuracy: 0.9969 - val_loss: 0.0359 - val_accuracy: 0.9910
Epoch 3/3
1688/1688 [==============================] - 10s 6ms/step - loss: 0.0089 - accuracy: 0.9978 - val_loss: 0.0346 - val_accuracy: 0.9915


## Sparsity 확인

In [ ]:
def print_model_weights_sparsity(model):
    for layer in model.layers:
        if isinstance(layer, keras.layers.Wrapper):
            weights = layer.trainable_weights
        else:
            weights = layer.weights
        for weight in weights:
            if "quantize_layer" in weight.name:
                continue
            weight_size = weight.numpy().size
            zero_num = np.count_nonzero(weight == 0)
            print(
                f"{weight.name}: {zero_num/weight_size:.2%} sparsity ",
                f"({zero_num}/{weight_size})",
            )

In [ ]:
stripped_pruned_model = tfmot.sparsity.keras.strip_pruning(pruned_model)

print_model_weights_sparsity(stripped_pruned_model)

conv2d/kernel:0: 50.00% sparsity  (144/288)
conv2d/bias:0: 0.00% sparsity  (0/32)
conv2d_1/kernel:0: 50.00% sparsity  (2304/4608)
conv2d_1/bias:0: 0.00% sparsity  (0/16)
dense/kernel:0: 50.00% sparsity  (25600/51200)
dense/bias:0: 0.00% sparsity  (0/128)
dense_1/kernel:0: 50.00% sparsity  (640/1280)
dense_1/bias:0: 0.00% sparsity  (0/10)


## Accuracy Check : Baseline model VS Pruned model

In [ ]:
_, pruned_model_accuracy = pruned_model.evaluate(
  test_images, test_labels, verbose=0)

print('Baseline test accuracy:', baseline_model_accuracy)
print('Pruned test accuracy:', pruned_model_accuracy)

Baseline test accuracy: 0.9904000163078308
Pruned test accuracy: 0.9908000230789185


## QAT VS PQAT
* QAT : training 과정에서 sparsity 파괴
* PQAT : training 과정에서도 sparsity 유지

In [ ]:
# QAT
qat_model = tfmot.quantization.keras.quantize_model(stripped_pruned_model)

qat_model.compile(optimizer='adam',
              loss=keras.losses.SparseCategoricalCrossentropy(),
              metrics=['accuracy'])
print('Train QAT model:')
qat_model.fit(train_images, train_labels, batch_size=128, epochs=1, validation_split=0.1)

Train QAT model:
422/422 [==============================] - 7s 8ms/step - loss: 0.0063 - accuracy: 0.9982 - val_loss: 0.0386 - val_accuracy: 0.9925


In [ ]:
# PQAT
quant_aware_annotate_model = tfmot.quantization.keras.quantize_annotate_model(
              stripped_pruned_model)
pqat_model = tfmot.quantization.keras.quantize_apply(
              quant_aware_annotate_model,
              tfmot.experimental.combine.Default8BitPrunePreserveQuantizeScheme())

pqat_model.compile(optimizer='adam',
              loss=keras.losses.SparseCategoricalCrossentropy(),
              metrics=['accuracy'])
print('Train PQAT Model:')
pqat_model.fit(train_images, train_labels, batch_size=128, epochs=1, validation_split=0.1)

Train PQAT Model:
422/422 [==============================] - 6s 9ms/step - loss: 0.0056 - accuracy: 0.9988 - val_loss: 0.0390 - val_accuracy: 0.9920


## Sparsity 확인 : QAT model VS PQAT model

In [ ]:
print("PQAT Model sparsity:")
print_model_weights_sparsity(pqat_model)
print()
print("QAT Model sparsity:")
print_model_weights_sparsity(qat_model)

PQAT Model sparsity:
conv2d/kernel:0: 50.00% sparsity  (144/288)
conv2d/bias:0: 0.00% sparsity  (0/32)
conv2d_1/kernel:0: 50.00% sparsity  (2304/4608)
conv2d_1/bias:0: 0.00% sparsity  (0/16)
dense/kernel:0: 50.00% sparsity  (25600/51200)
dense/bias:0: 0.00% sparsity  (0/128)
dense_1/kernel:0: 50.00% sparsity  (640/1280)
dense_1/bias:0: 0.00% sparsity  (0/10)

QAT Model sparsity:
conv2d/kernel:0: 7.99% sparsity  (23/288)
conv2d/bias:0: 0.00% sparsity  (0/32)
conv2d_1/kernel:0: 5.69% sparsity  (262/4608)
conv2d_1/bias:0: 0.00% sparsity  (0/16)
dense/kernel:0: 11.73% sparsity  (6008/51200)
dense/bias:0: 0.00% sparsity  (0/128)
dense_1/kernel:0: 6.72% sparsity  (86/1280)
dense_1/bias:0: 0.00% sparsity  (0/10)


## LiteRT 모델로 변환 (QAT)

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(qat_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/convert.py:854: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


In [ ]:
tflite_qat_file = save_dir + 'mnist_pqat_x.tflite'
open(tflite_qat_file, 'wb').write(tflite_model)

63888

## LiteRT 모델로 변환 (PQAT)

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(pqat_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/convert.py:854: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


In [ ]:
tflite_pqat_file = save_dir + 'mnist_pqat.tflite'
open(tflite_pqat_file, 'wb').write(tflite_model)

63888

## 모델 압축 테스트

* 압축 함수 정의

In [ ]:
import zipfile
import os

def get_gzipped_model_size(file):
  # It returns the size of the gzipped model in kilobytes.

  _, zipped_file = tempfile.mkstemp('.zip')
  with zipfile.ZipFile(zipped_file, 'w', compression=zipfile.ZIP_DEFLATED) as f:
    f.write(file)

  return os.path.getsize(zipped_file)/1000

* 압축된 파일 크기 비교

In [ ]:
print("QAT model size: ", get_gzipped_model_size(tflite_qat_file), ' KB')
print("PQAT model size: ", get_gzipped_model_size(tflite_pqat_file), ' KB')

QAT model size:  46.917  KB
PQAT model size:  38.98  KB


## LiteRT 설치 및 Interpreter 로딩

In [ ]:
!pip install ai-edge-litert

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.6/9.6 MB 80.9 MB/s eta 0:00:00


In [ ]:
from ai_edge_litert.interpreter import Interpreter

## interpreter 생성 (PQAT)

In [ ]:
interpreter_pqat = Interpreter(model_path=str(tflite_pqat_file))
interpreter_pqat.allocate_tensors()

## input/output dtype 확인 (PQAT)

In [ ]:
input_dtype = interpreter_pqat.get_input_details()[0]['dtype']
output_dtype = interpreter_pqat.get_output_details()[0]['dtype']

print("input dtype : {}".format(input_dtype))
print("output dtype : {}".format(output_dtype))

input dtype : <class 'numpy.float32'>
output dtype : <class 'numpy.float32'>


## Test data 기반 accuracy 평가

In [ ]:
def eval_model(interpreter):
  input_details = interpreter.get_input_details()[0]
  output_details = interpreter.get_output_details()[0]
  input_index = input_details["index"]
  output_index = output_details["index"]

  prediction_digits = []
  for i, test_image in enumerate(test_images):
    test_image = np.expand_dims(test_image, axis=0).astype(input_details['dtype'])
    interpreter.set_tensor(input_index, test_image)

    interpreter.invoke()

    output = interpreter.get_tensor(output_index)
    digit = np.argmax(output)
    prediction_digits.append(digit)

  prediction_digits = np.array(prediction_digits)
  accuracy = (prediction_digits == test_labels).mean()
  return accuracy

In [ ]:
pqat_test_accuracy = eval_model(interpreter_pqat)

print('Clustered and quantized TFLite test_accuracy :', pqat_test_accuracy)
print('Baseline model test_accuracy :', baseline_model_accuracy)

Clustered and quantized TFLite test_accuracy : 0.9926
Baseline model test_accuracy : 0.9904000163078308
